# `pgvector_retriever` (PgVectorRetrieverModule)
- **Directory**: `modules/retrieval/pgvector_retriever.py`
- **Category**: Logic (Dense Retrieval)
- **Role**: 서브쿼리 임베딩 벡터와 pgvector 인덱스 간 코사인 유사도를 계산하여 Top-K 밀집 검색 후보를 반환합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().parent.name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.retrieval.pgvector_retriever import PgVectorRetrieverModule, PgVectorRetrieverInputDTO, PgVectorRetrieverConfigDTO

mock_store = MagicMock()
mock_doc = MagicMock()
mock_doc.page_content = "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: 65670"
mock_doc.metadata = {"cell_id": "삼성전자:손익계산서:D5"}
mock_store.similarity_search_by_vector_with_score.return_value = [(mock_doc, 0.05)]

module = PgVectorRetrieverModule(pgvector_store=mock_store)

sample_input = {
    "query_input": {
        "query_context": {"question_id": "QUERY-001", "question_text": "2023년 삼성전자 영업이익"},
        "items": {
            "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: ?": [0.05] * 3072
        }
    },
    "index_input": {
        "index_id": "idx_samsung_2023",
        "file_name": "samsung_2023.xlsx",
        "workbook_hash": "hash_samsung_2023",
        "model": "text-embedding-3-large",
        "dimension": 3072,
        "document_count": 1000
    }
}
input_dto = PgVectorRetrieverInputDTO(**sample_input)
output = module.run(input_dto, config=PgVectorRetrieverConfigDTO(top_k=5))
print_io("pgvector_retriever (PgVectorRetrieverModule)", sample_input, output)
